In [12]:
# These are the libraries you may use for this assignment.
# If you would like to use any other libraries, email james@uwaterloo.ca for permission first.
# Generative AI tools (including coding assistants) are not permitted for this assignment.

from IPython.display import display, Math, Latex
%pip install pandas
%pip install numpy_financial
%pip install yfinance
%pip install matplotlib
import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 20.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 22.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 29.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [matplotlib]8 [matplotlib]esources]
Note: you may need to restart the kernel to use updated packages.


Matplotlib is building the font cache; this may take a moment.


## CFM 101 — Group Assignment 2026
### Robo-Advising Challenge: Two Clients, Two Strategies

**Team Number:** 12
**Team Member Names (three or four):**  

Enter the names of all participating members below (one name per line):

Daniel Lei
Turki 
Ruoxi
Grace

**Submission:** Friday, November 27, 2026, at 9:00 AM (Eastern), via Dropbox.  
**Submit this completed Jupyter notebook only.**

Your team must construct **both** portfolios from the supplied `Tickers.csv` file:

- **Client A — Growth:** $\$1,000,000$ CAD; seek the highest expected return.
- **Client B — Low Risk:** $\$750,000$ CAD; seek the lowest portfolio volatility.

Both clients require 10–20 eligible stocks, at most 15% in any stock and at most 40% in any sector. Read the full assignment handout for all eligibility, fee, currency, and competition rules. Do not hardcode the selected stocks. Check that tickers are valid. The supplied ticker universe will permit feasible portfolios.

**No generative AI may be used in preparing this assignment.**

## Data preparation and shared functions

Read `Tickers.csv`, check ticker eligibility, retrieve the data needed for your strategies, and create any functions you will reuse for both clients. Use closing prices from **November 27, 2026** for the purchase calculations. All portfolio values and weights must be calculated in CAD, with purchase fees paid from the client budget.

here is a test - hello there why is it not working oooooooooo

pls work




wowoowowowowowo

In [14]:

tickers_df = pd.read_csv('Tickers_Example.csv', header = None, dtype=str)

start_date = '2025-10-01'
end_date = '2026-10-01'

ticker_list = []
for ticker in tickers_df.iloc[:, 0].dropna():
    ticker = ticker.replace(' ', '').upper()
    # skip blanks any title rows and duplicates (e.g. RY.TO listed twice)
    if ticker == '' or ticker == 'TICKER' or ticker in ticker_list:
        continue

    ticker_list.append(ticker)

In [15]:
symbols = ticker_list + ['^GSPC', 'CAD=X']

close = pd.DataFrame()
volume = pd.DataFrame()
for attempt in range(3):
    missing = [t for t in symbols if t not in close.columns]
    if len(missing) == 0:
        break
    data = yf.download(missing, start=start_date, end=end_date, auto_adjust=False, progress=False)
    if data.empty:
        continue
    new_close = data['Close'].dropna(axis=1, how='all')
    close = pd.concat([close, new_close], axis=1)
    volume = pd.concat([volume, data['Volume'][new_close.columns]], axis=1)

print('No price data (invalid or delisted):', [t for t in ticker_list if t not in close.columns])

$BK: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)
$CELG: possibly delisted; no timezone found
$RTN: possibly delisted; no timezone found
$AGN: possibly delisted; no timezone found
$MON: possibly delisted; no timezone found

5 Failed downloads:
['BK']: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)
['CELG', 'RTN', 'AGN', 'MON']: possibly delisted; no timezone found
$AGN: possibly delisted; no timezone found
$RTN: possibly delisted; no timezone found
$MON: possibly delisted; no timezone found
$CELG: possibly delisted; no timezone found
$BK: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)

5 Failed downloads:
['AGN', 'RTN', 'MON', 'CELG']: possibly delisted; no timezone found
['BK']: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)
$AGN: possibly delisted; no timezone found
$RTN: possibly delisted; no timezone found
$MON: possibly delisted; no timezone found
$CELG: possibly delisted; no timezone fou

No price data (invalid or delisted): ['AGN', 'BK', 'CELG', 'MON', 'RTN']


In [16]:
stock_volume = volume.drop(columns=['^GSPC', 'CAD=X'], errors='ignore')

# months with fewer than 18 trading days are ignored
days_per_month = stock_volume.groupby(stock_volume.index.to_period('M')).transform('count')
stock_volume = stock_volume.where(days_per_month >= 18)

mean_volumes = stock_volume.mean()
filtered_tickers = mean_volumes[mean_volumes >= 5000].index.tolist()
filtered_tickers

['AAPL',
 'ABBV',
 'ABT',
 'ACN',
 'AIG',
 'AMZN',
 'AXP',
 'BA',
 'BAC',
 'BB.TO',
 'BIIB',
 'BLK',
 'BMY',
 'C',
 'CAT',
 'CL',
 'KO',
 'LLY',
 'LMT',
 'MO',
 'MRK',
 'PEP',
 'PFE',
 'PM',
 'PYPL',
 'QCOM',
 'RY.TO',
 'SHOP.TO',
 'T.TO',
 'TD.TO',
 'TXN',
 'UNH',
 'UNP',
 'UPS',
 'USB']

In [20]:
allowed_exchanges = ['NMS', 'NGM', 'NCM', 'NYQ', 'NYS', 'ASE', 'PCX', 'BTS', 'BATS', 'NAS', 'TOR', 'VAN', 'CNQ', 'NEO']

info_stocks = {}
for ticker in filtered_tickers:
    info = yf.Ticker(ticker).get_info()
    if info.get('quoteType') != 'EQUITY':
        print(f'Removed (not a stock): {ticker}')
    elif info.get('currency') not in ['CAD', 'USD']:
        print(f'Removed (currency is {info.get("currency")}): {ticker}')
    elif info.get('exchange') not in allowed_exchanges:
        print(f'Removed (not listed in Canada/US, exchange is {info.get("exchange")}): {ticker}')
    else:
        info_stocks[ticker] = {'Name': info.get('shortName', ticker),
                               'Sector': info.get('sector') or 'Unknown',
                               'Currency': info['currency']}

info_stocks = pd.DataFrame(info_stocks).T
filtered_tickers = info_stocks.index.tolist()
filtered_tickers

['AAPL',
 'ABBV',
 'ABT',
 'ACN',
 'AIG',
 'AMZN',
 'AXP',
 'BA',
 'BAC',
 'BB.TO',
 'BIIB',
 'BLK',
 'BMY',
 'C',
 'CAT',
 'CL',
 'KO',
 'LLY',
 'LMT',
 'MO',
 'MRK',
 'PEP',
 'PFE',
 'PM',
 'PYPL',
 'QCOM',
 'RY.TO',
 'SHOP.TO',
 'T.TO',
 'TD.TO',
 'TXN',
 'UNH',
 'UNP',
 'UPS',
 'USB']

In [18]:
seen = {}
for t in filtered_tickers.copy():
    key = info_stocks.loc[t, 'Name'].lower().replace('.', '').replace(',', '').strip()
    if key not in seen:
        seen[key] = t
    elif mean_volumes[t] > mean_volumes[seen[key]]:
        print(f'Removed (second listing of the same company): {seen[key]}')
        filtered_tickers.remove(seen[key])
        seen[key] = t
    else:
        print(f'Removed (second listing of the same company): {t}')
        filtered_tickers.remove(t)

info_stocks = info_stocks.loc[filtered_tickers]
filtered_tickers

['AAPL',
 'ABBV',
 'ABT',
 'ACN',
 'AIG',
 'AMZN',
 'AXP',
 'BA',
 'BAC',
 'BB.TO',
 'BIIB',
 'BLK',
 'BMY',
 'C',
 'CAT',
 'CL',
 'KO',
 'LLY',
 'LMT',
 'MO',
 'MRK',
 'PEP',
 'PFE',
 'PM',
 'PYPL',
 'QCOM',
 'RY.TO',
 'SHOP.TO',
 'T.TO',
 'TD.TO',
 'TXN',
 'UNH',
 'UNP',
 'UPS',
 'USB']

In [82]:
temp_list = ticker_data['Close']
ticker_list = temp_list.columns.tolist()
ticker_list


NameError: name 'ticker_data' is not defined

In [22]:
fx = close['CAD=X'].ffill()
market = close['^GSPC'].ffill() * fx

prices = close[filtered_tickers].ffill()
for t in filtered_tickers:
    if info_stocks.loc[t, 'Currency'] == 'USD':
        prices[t] = prices[t] * fx

returns = prices.pct_change().iloc[1:]
market_returns = market.pct_change().iloc[1:]

beta = {}
for t in filtered_tickers:
    beta[t] = returns[t].cov(market_returns) / market_returns.var()
beta = pd.Series(beta)

standard_deviation = returns.std()
average_return = returns.mean()

pd.DataFrame({'Beta': beta, 'Daily std': standard_deviation, 'Avg daily return': average_return})

,Beta,Daily std,Avg daily return
AAPL,0.736774,0.015533,0.001223
ABBV,0.052005,0.016383,0.000471
ABT,0.132084,0.016980,-0.000947
ACN,0.337659,0.028184,-0.000625
AIG,0.102999,0.015198,-0.000074
AMZN,1.367508,0.021372,0.000768
AXP,0.998835,0.016921,-0.000082
BA,1.293161,0.021654,-0.000258
BAC,0.761664,0.014034,0.000449
BB.TO,1.406293,0.037817,0.003238


After removing all the unwanted stocks, get data for each of the remaining stocks that will help us determine if they are stable and optimal for us to add to our portfolio:
Standard deviation
Tells us how volatile/risky the stock is, and we will look for stocks that have lower standard deviations (close to 0) to reduce the volatility and risk of our portfolio
Beta to the S&P 500
Tells us how closely the stock moves with the market. We want to look for stocks that have lower betas (close to 0 in absolute value) because it will mean that the stock's movements aren't very correlated with the market, and don't particularly go up or down when the market moves
Expected Monthly Returns
Tells us what percentage return we should expect to see per month. We want to look for stocks that have lower expected monthly returns (close to 0 in absolute value) because we want a portfolio that gives as close to 0% returns as possible

| Signal | Why it points toward higher return | Weight |
|---|---|---|
| **Beta** (Chapter 15) | In a rising market, a beta-1.5 stock tends to move about 1.5× the market. CAPM: more market risk earns more expected return. | 40% |
| **12-1 month momentum** | Past-year winners (skipping the latest month, which tends to bounce back) have tended to keep winning (Jegadeesh & Titman, 1993). Research evidence, not from our slides. | 40% |
| **Revenue growth** (Yahoo) | Our client wants *growth*; fast-growing sales support a company's story. Yahoo's data is patchy, so it gets less weight and **a missing value counts as neutral**. | 20% |

**Selection:** top 12 by score, at most 4 per sector. **Weighting:** proportional to score (shifted so the lowest of the 12 is still positive),
then the 15% / 40% caps are applied and any excess is spread over the uncapped stocks.
**What we deliberately did *not* use:** dividends — the competition ignores dividends, so they earn nothing in this contest.

### Client A — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.
Client A aims for the maximum return and is willing to take high risks. As expected returns cannot be observed, I have measured each stock against three factors and aggregated them using their z-scores (their distance from the mean of the entire set). Beta will account for 40%: stock with beta greater than 1 will move even more upwards than the movement of the market, and hence, the theory suggests that investors get higher compensation for market risk according to CAPM. Momentum over 12-1 month period will get 40%. 20% for revenue growth that corresponds to the growth strategy, although it is given less weight since the data for Yahoo is unreliable and the absence of a figure means a neutral score. I have purchased the top 12 based on their scores, where each sector is allowed a maximum of 4 stocks so that no single sector will be dominant. This was done intentionally since the contest gives recognition solely based on the best performing stock over five days, and with fewer stocks, there is greater variation in performance. Higher scores receive larger allocations with the stock limit of 15% and the sector limit of 40%. 

Client A is likely high-scoring (higher beta, growth, strong past year): so stocks like 
AMZN, SHOP.TO, QCOM, LLY, PYPL, AXP, CAT, BA, plus possibly C, BAC, TXN and ACN. BB.TO is very volatile and could score high or low depending on its momentum. The 4-per-sector limit will push the list to spread across tech, financials, industrials and healthcare.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.
If the market falls between Nov 30 and Dec 4, high-beta stocks will fall more than the market, so the beta tilt works against us. Momentum can reverse suddenly, and over only five days luck matters more than any signal. Up to 40% of the portfolio can sit in one sector, so a sector-specific shock would hurt. US stocks also lose value in CAD if the Canadian dollar strengthens.

In [29]:
ss = yf.Ticker('NVDA')
ss.info

{'address1': '2788 San Tomas Expressway',
 'city': 'Santa Clara',
 'state': 'CA',
 'zip': '95051',
 'country': 'United States',
 'phone': '408 486 2000',
 'website': 'https://www.nvidia.com',
 'industry': 'Semiconductors',
 'industryKey': 'semiconductors',
 'industryDisp': 'Semiconductors',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': "NVIDIA Corporation operates as a data center scale AI infrastructure company in the United States, Taiwan, China, Hong Kong, Europe, and internationally. It operates through Compute & Networking, and Graphics segments. The Compute & Networking segment provides data center accelerated computing and networking platforms and artificial intelligence solutions and software, and automotive platforms and autonomous and electric vehicle solutions, including software. The Graphics segment offers GeForce GPUs for gaming and PCs; Quadro/NVIDIA RTX GPUs for enterprise workstation graphics. The company's pr

In [66]:

data = []

for i in filtered_tickers:
    stock = yf.Ticker(i)
    info = stock.info
    data.append([i,info.get('revenueGrowth'),info.get('earningsGrowth'),info.get('returnOnAssets'),info.get('profitMargins'),info.get
                 ('trailingPE'),('beta)'),('Daily std')])



In [72]:
val_criteria = pd.DataFrame(data, columns=['Ticker', 'Revenue Growth', 'Earning Growth', 'Return on Assets', 'Profit Margins', 'Trailing PE', 'Beta', 'Daily Std'])
val_criteria


,Ticker,Revenue Growth,Earning Growth,Return on Assets,Profit Margins,Trailing PE,Beta,Daily Std
0,AAPL,0.117,0.141,0.25868,0.27604,38.531185,beta),Daily std
1,ABBV,0.103,22.821,0.10736,0.15666,49.635740,beta),Daily std
2,ABT,0.145,-0.436,0.04902,0.09747,38.850746,beta),Daily std
3,ACN,0.062,0.464,0.10954,0.11279,16.037209,beta),Daily std
4,AIG,0.164,0.720,0.01945,0.11838,13.425496,beta),Daily std
5,AMZN,0.121,-0.031,0.06618,0.16901,20.364996,beta),Daily std
6,AXP,0.113,0.309,0.04033,0.16823,17.384226,beta),Daily std
7,BA,0.039,NaN,-0.00108,0.08066,16.487100,beta),Daily std
8,BAC,0.034,-0.125,0.00944,0.28064,11.420047,beta),Daily std
9,BB.TO,0.260,1.500,0.05026,0.13094,66.117645,beta),Daily std


In [ ]:

#since Yfinance dont provide earning growth so we will calculate them manually using the quarter income statement and the 
# net income of the last two quarters.

stock1 = yf.Ticker('QCOM')
income = stock1.quarterly_income_stmt
ear = income.loc['Net Income']
now = ear.iloc[0]
previous = ear.iloc[1]



growth = (now - previous) / (previous)
print(growth)

stock3 = yf.Ticker('BA')
income = stock3.quarterly_income_stmt
ear = income.loc['Net Income']
now = ear.iloc[0]
previous = ear.iloc[1]
#since its provides nan we will consider it 0


growth = (now - previous) / (previous)
print(growth)

stock2 = yf.Ticker('T.TO')
income = stock2.quarterly_income_stmt
ear = income.loc['Net Income']
now = ear.iloc[0]
previous = ear.iloc[1]



growth = (now - previous) / (previous)
print(growth)



-0.37512487512487513
nan
-0.6375


In [98]:
r_rev = []
for x in val_criteria['Revenue Growth']:
    r_rev_gro = 1
    for y in val_criteria['Revenue Growth']:
        if y > x:
            r_rev_gro = r_rev_gro + 1
    r_rev.append(r_rev_gro)
val_criteria['Revenue Growth Rank'] = r_rev
val_criteria

,Ticker,Revenue Growth,Earning Growth,Return on Assets,Profit Margins,Trailing PE,Beta,Daily Std,Revenue Growth Rank
0,AAPL,0.117,0.141,0.25868,0.27604,38.531185,beta),Daily std,11
1,ABBV,0.103,22.821,0.10736,0.15666,49.635740,beta),Daily std,16
2,ABT,0.145,-0.436,0.04902,0.09747,38.850746,beta),Daily std,9
3,ACN,0.062,0.464,0.10954,0.11279,16.037209,beta),Daily std,23
4,AIG,0.164,0.720,0.01945,0.11838,13.425496,beta),Daily std,7
5,AMZN,0.121,-0.031,0.06618,0.16901,20.364996,beta),Daily std,10
6,AXP,0.113,0.309,0.04033,0.16823,17.384226,beta),Daily std,13
7,BA,0.039,NaN,-0.00108,0.08066,16.487100,beta),Daily std,28
8,BAC,0.034,-0.125,0.00944,0.28064,11.420047,beta),Daily std,30
9,BB.TO,0.260,1.500,0.05026,0.13094,66.117645,beta),Daily std,4


In [101]:
r_ret = []
for x in val_criteria['Return on Assets']:
    r_ret_ast = 1
    for y in val_criteria['Return on Assets']:
        if y > x:
            r_ret_ast = r_ret_ast + 1
    r_ret.append(r_ret_ast)
val_criteria['Return on assets Rank'] = r_ret
val_criteria

,Ticker,Revenue Growth,Earning Growth,Return on Assets,Profit Margins,Trailing PE,Beta,Daily Std,Revenue Growth Rank,Return on assets Rank,Earnings GrowthRank
0,AAPL,0.117,0.141,0.25868,0.27604,38.531185,beta),Daily std,11,1,11
1,ABBV,0.103,22.821,0.10736,0.15666,49.635740,beta),Daily std,16,11,16
2,ABT,0.145,-0.436,0.04902,0.09747,38.850746,beta),Daily std,9,24,9
3,ACN,0.062,0.464,0.10954,0.11279,16.037209,beta),Daily std,23,8,23
4,AIG,0.164,0.720,0.01945,0.11838,13.425496,beta),Daily std,7,29,7
5,AMZN,0.121,-0.031,0.06618,0.16901,20.364996,beta),Daily std,10,19,10
6,AXP,0.113,0.309,0.04033,0.16823,17.384226,beta),Daily std,13,26,13
7,BA,0.039,NaN,-0.00108,0.08066,16.487100,beta),Daily std,28,35,28
8,BAC,0.034,-0.125,0.00944,0.28064,11.420047,beta),Daily std,30,32,30
9,BB.TO,0.260,1.500,0.05026,0.13094,66.117645,beta),Daily std,4,22,4


In [102]:
r_ear = []
for x in val_criteria['Earning Growth']:
    r_ear_gro = 1
    for y in val_criteria['Earning Growth']:
        if y > x:
            r_ear_gro = r_ear_gro + 1
    r_ear.append(r_ear_gro)
val_criteria['Earnings GrowthRank'] = r_ear
val_criteria

,Ticker,Revenue Growth,Earning Growth,Return on Assets,Profit Margins,Trailing PE,Beta,Daily Std,Revenue Growth Rank,Return on assets Rank,Earnings GrowthRank
0,AAPL,0.117,0.141,0.25868,0.27604,38.531185,beta),Daily std,11,1,17
1,ABBV,0.103,22.821,0.10736,0.15666,49.635740,beta),Daily std,16,11,1
2,ABT,0.145,-0.436,0.04902,0.09747,38.850746,beta),Daily std,9,24,31
3,ACN,0.062,0.464,0.10954,0.11279,16.037209,beta),Daily std,23,8,8
4,AIG,0.164,0.720,0.01945,0.11838,13.425496,beta),Daily std,7,29,6
5,AMZN,0.121,-0.031,0.06618,0.16901,20.364996,beta),Daily std,10,19,24
6,AXP,0.113,0.309,0.04033,0.16823,17.384226,beta),Daily std,13,26,11
7,BA,0.039,NaN,-0.00108,0.08066,16.487100,beta),Daily std,28,35,1
8,BAC,0.034,-0.125,0.00944,0.28064,11.420047,beta),Daily std,30,32,30
9,BB.TO,0.260,1.500,0.05026,0.13094,66.117645,beta),Daily std,4,22,2


In [104]:

r_prof = []
for x in val_criteria['Profit Margins']:
    r_prof_marg = 1
    for y in val_criteria['Profit Margins']:
        if y > x:
            r_prof_marg = r_prof_marg + 1
    r_prof.append(r_prof_marg)
val_criteria['Profit Margins Rank'] = r_prof
val_criteria

,Ticker,Revenue Growth,Earning Growth,Return on Assets,Profit Margins,Trailing PE,Beta,Daily Std,Revenue Growth Rank,Return on assets Rank,Earnings GrowthRank,Profit Margins Rank
0,AAPL,0.117,0.141,0.25868,0.27604,38.531185,beta),Daily std,11,1,17,10
1,ABBV,0.103,22.821,0.10736,0.15666,49.635740,beta),Daily std,16,11,1,18
2,ABT,0.145,-0.436,0.04902,0.09747,38.850746,beta),Daily std,9,24,31,26
3,ACN,0.062,0.464,0.10954,0.11279,16.037209,beta),Daily std,23,8,8,24
4,AIG,0.164,0.720,0.01945,0.11838,13.425496,beta),Daily std,7,29,6,23
5,AMZN,0.121,-0.031,0.06618,0.16901,20.364996,beta),Daily std,10,19,24,16
6,AXP,0.113,0.309,0.04033,0.16823,17.384226,beta),Daily std,13,26,11,17
7,BA,0.039,NaN,-0.00108,0.08066,16.487100,beta),Daily std,28,35,1,29
8,BAC,0.034,-0.125,0.00944,0.28064,11.420047,beta),Daily std,30,32,30,9
9,BB.TO,0.260,1.500,0.05026,0.13094,66.117645,beta),Daily std,4,22,2,22


In [106]:

r_trail = []
for x in val_criteria['Trailing PE']:
    r_trail_pe = 1
    for y in val_criteria['Trailing PE']:
        if y > x:
            r_trail_pe = r_trail_pe + 1
    r_trail.append(r_trail_pe)
val_criteria['Trailing PE Rank'] = r_trail
val_criteria

,Ticker,Revenue Growth,Earning Growth,Return on Assets,Profit Margins,Trailing PE,Beta,Daily Std,Revenue Growth Rank,Return on assets Rank,Earnings GrowthRank,Profit Margins Rank,Trailing PE Rank
0,AAPL,0.117,0.141,0.25868,0.27604,38.531185,beta),Daily std,11,1,17,10,9
1,ABBV,0.103,22.821,0.10736,0.15666,49.635740,beta),Daily std,16,11,1,18,5
2,ABT,0.145,-0.436,0.04902,0.09747,38.850746,beta),Daily std,9,24,31,26,8
3,ACN,0.062,0.464,0.10954,0.11279,16.037209,beta),Daily std,23,8,8,24,26
4,AIG,0.164,0.720,0.01945,0.11838,13.425496,beta),Daily std,7,29,6,23,28
5,AMZN,0.121,-0.031,0.06618,0.16901,20.364996,beta),Daily std,10,19,24,16,17
6,AXP,0.113,0.309,0.04033,0.16823,17.384226,beta),Daily std,13,26,11,17,23
7,BA,0.039,NaN,-0.00108,0.08066,16.487100,beta),Daily std,28,35,1,29,25
8,BAC,0.034,-0.125,0.00944,0.28064,11.420047,beta),Daily std,30,32,30,9,31
9,BB.TO,0.260,1.500,0.05026,0.13094,66.117645,beta),Daily std,4,22,2,22,4


## Client B — Low Risk

Construct a portfolio intended to achieve the **lowest risk**, defined as the standard deviation of daily portfolio returns in CAD. Create `Portfolio_Final_B` using the same DataFrame column names, order, and index convention as for Client A.

### Client B — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

## Equal-weight comparisons

For **each client**, create an equal-weighted comparison portfolio holding **exactly the same stocks** as that client’s chosen portfolio, with starting weight `1/n` per stock. Over the same clearly identified historical period, calculate **historical return** and **volatility** for the chosen portfolio and its equal-weighted counterpart. Use consistent CAD valuations and weighting assumptions.

Display one concise comparison table per client, and interpret the results in the financial justifications above. No additional graph or out-of-sample test is required.

## Programmatic validation

Use Python (`assert` statements or a simple function are fine) to display whether **both** final portfolios pass each of the following checks:

- Selected tickers are valid and eligible.
- Number of stocks is between 10 and 20.
- No individual stock weight exceeds 15%.
- No single sector has more than 40% of portfolio value.
- Portfolio weights sum to 100%.
- Total stock purchases plus transaction fees do not exceed the applicable client budget.

Your code should run with the TAs’ ticker file without manually changing individual stock selections.

## Final portfolio outputs

Display `Portfolio_Final_A` and `Portfolio_Final_B`, along with the budget reconciliation (purchases plus fees) for each. Generate `Stocks_Group_XX_A.csv` and `Stocks_Group_XX_B.csv` (replace `XX` with your team number). Each CSV must contain **only** the columns `Ticker` and `Shares`. The notebook must **generate** these CSV files when run, but you submit **only the notebook** to Dropbox.

## Contribution and AI Declaration

**The following team members made a meaningful contribution to this assignment:**

Enter the names of all contributing members below (one name per line):

**To the best of our knowledge, no generative AI was used in preparing this submission.**

List only members who made a meaningful contribution. Contact the instructor before the deadline if someone should be omitted or if the team cannot make the declaration.